# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/hammadkhaliq-del/flyrank-ml-internship-starter/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*

**Task type: scoring (feeding a ranking), not classification or clustering.**

Lane 2's job is to hand a content editor an ordered review queue, top N pages first. A **classification** model (declining / not declining) stops at a yes/no flag — it was already tried in Week 1 as the plain rule (`trend_direction == "down"`) and it tags 54.2% of the inventory identically, with no way to say which of those pages to open first. **Clustering** would group similar pages together but wouldn't tell an editor which cluster to work first either. What the decision actually needs is a **continuous score per page** that can be sorted — that's a scoring/regression-shaped problem whose output feeds a ranking, not a class label or a group ID.

The code cell below makes this concrete: it shows the single most "obviously important" bucket a classifier would produce (declining + real demand + page-1 ranking), and how many pages land inside it tied at the same priority with no further order — which is exactly the gap a scoring approach closes.

In [1]:
import pandas as pd

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

# The naive version of this task is binary classification: "is this page declining, yes/no?"
# But a yes/no flag can't be handed to an editor -- it still needs an ORDER.
# Show why: how many pages fall into the single most valuable rule-based bucket,
# all tied at the same "priority" under a classifier with no ranking inside it?
declining = df[df["trend_direction"] == "down"]
top_bucket = declining[
    (declining["position_tier"].isin(["page_1", "top_3"])) &
    (declining["impressions_90d"] >= 300)
]
print(f"Pages a binary classifier would tag identically as 'top priority': {len(top_bucket):,}")
print("Inside that single tied bucket, demand still spans:")
print(f"  impressions_90d: {top_bucket['impressions_90d'].min():.0f} to {top_bucket['impressions_90d'].max():,.0f}")
print(f"  avg_position:    {top_bucket['avg_position'].min():.1f} to {top_bucket['avg_position'].max():.1f}")
print()
print("A classifier stops here. An editor still needs to know which of these ~4,900")
print("pages to open FIRST -- that need for an internal order is what makes this a")
print("SCORING / RANKING task, not classification or clustering.")


Pages a binary classifier would tag identically as 'top priority': 4,872
Inside that single tied bucket, demand still spans:
  impressions_90d: 300 to 517,715
  avg_position:    0.2 to 10.0

A classifier stops here. An editor still needs to know which of these ~4,900
pages to open FIRST -- that need for an internal order is what makes this a
SCORING / RANKING task, not classification or clustering.


## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

**Target: `priority_score`, a proxy score from 0–100, not an observed ground-truth outcome.**

There is no column in this dataset that says "an editor reviewed this page and it was worth it" — no such outcome was ever logged, so this cannot be a fully observed label the way, say, "did the page get more clicks next month" would be. It's a **proxy** built from three *observed* 90-day signals, combined:

1. **`is_declining`** — from `trend_direction == "down"`, itself derived from a real 30-day vs. 60-day impression comparison (observed, not opinion).
2. **`demand_component`** — how much real search demand the page still pulls (`impressions_90d`, log-scaled so a handful of huge outlier pages don't dominate).
3. **`position_component`** — how good the page's current ranking is, i.e., how much value there is to protect if it keeps declining.

`priority_score = is_declining × (0.5 × demand_component + 0.5 × position_component) × 100`

Non-declining pages score 0 by construction — the queue is only ever built from pages that are actually declining. Being explicit about this being a **proxy** (not ground truth) matters: if the internship's Week 3+ warehouse release later has an actual editor-action log, that would become the real target to predict and this proxy would become the baseline to beat.

In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

# --- Build the proxy target: priority_score (0-100) ---
# This is a PROXY, not a ground-truth label, because no editor-outcome log exists
# in this dataset (no "editor reviewed this and it was worth it" column).
# It's built entirely from OBSERVED 90-day performance, not a hand-picked cutoff:
#   1. is_declining: observed rule from trend_direction (itself built from a
#      30-day vs 60-day impression comparison -- an outcome, not an opinion)
#   2. demand_component: how much real search demand the page still has,
#      log-scaled and clipped to 0-1 so a handful of huge outlier pages
#      don't swamp everything else
#   3. position_component: how good the page's current ranking is, i.e. how much
#      there is to protect (page-1 pages have more to lose than page-5 pages)
is_declining = (df["trend_direction"] == "down").astype(int)

demand_component = np.log1p(df["impressions_90d"].clip(lower=0))
demand_component = (demand_component - demand_component.min()) / (demand_component.max() - demand_component.min())

position_value_map = {"top_3": 1.0, "page_1": 0.85, "striking": 0.55, "page_3_5": 0.25, "deep": 0.05}
position_component = df["position_tier"].map(position_value_map).fillna(0.05)

df["priority_score"] = (is_declining * (0.5 * demand_component + 0.5 * position_component) * 100).round(1)

print("priority_score summary (0 = not worth a look, 100 = drop everything and review):")
print(df["priority_score"].describe().round(2))
print()
print("Distribution by decline status:")
print(df.groupby(is_declining)["priority_score"].describe()[["count","mean","max"]].round(2))
print()
print("Top 5 pages by priority_score:")
print(df.sort_values("priority_score", ascending=False)[
    ["content_id","trend_direction","position_tier","impressions_90d","priority_score"]
].head(5).to_string(index=False))


priority_score summary (0 = not worth a look, 100 = drop everything and review):
count    30000.00
mean        29.18
std         29.30
min          0.00
25%          0.00
50%         30.70
75%         55.40
max         99.90
Name: priority_score, dtype: float64

Distribution by decline status:
                   count   mean   max
trend_direction                      
0                13738.0   0.00   0.0
1                16262.0  53.84  99.9

Top 5 pages by priority_score:
          content_id trend_direction position_tier  impressions_90d  priority_score
content_8c19996aa890            down         top_3           509252            99.9
content_4c36c775b818            down         top_3           463103            99.6
content_9532f197bbc8            down         top_3           309192            97.9
content_5fe46e04994d            down        page_1           517715            92.5
content_1a9e894be2e2            down        page_1           416180            91.6


## 3. Success metric

*One metric you can defend. What number means 'good'?*

**Metric: precision@K on the ranked queue** — the share of the top-K pages by `priority_score` that are "truly worth reviewing."

This matches the cost structure named in Week 1: editor hours are the scarce resource, so a wasted review (false positive at the top of the queue) is the expensive, immediate mistake — precision at the top of the list is what protects that. Since no real editor-outcome log exists to validate against, I use an **independent proxy** for "truly worth reviewing": the same three-signal bucket from Week 1 (declining AND `impressions_90d >= 300` AND `position_tier` in page-1/top-3). It's still a proxy, but it wasn't built the same way as `priority_score` (no log-scaling, no weighted blend), so checking one against the other is a real test, not circular.

**What "good" looks like:** precision@K should be far above the base rate (16.2% of all pages are "truly worth reviewing"), and it should stay high through as large a K as possible before decaying — the code cell shows it's 100% through K=2,000 and only starts falling around K≈5,000–6,000, which is where the queue runs out of the highest-confidence pages and starts including partial matches. That decay point is itself useful — it tells an editor roughly how deep a "high-confidence" queue actually runs.

In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

is_declining = (df["trend_direction"] == "down").astype(int)
demand_component = np.log1p(df["impressions_90d"].clip(lower=0))
demand_component = (demand_component - demand_component.min()) / (demand_component.max() - demand_component.min())
position_value_map = {"top_3": 1.0, "page_1": 0.85, "striking": 0.55, "page_3_5": 0.25, "deep": 0.05}
position_component = df["position_tier"].map(position_value_map).fillna(0.05)
df["priority_score"] = (is_declining * (0.5 * demand_component + 0.5 * position_component) * 100).round(1)

# "Ground truth" for evaluation purposes: since no real editor-action log exists,
# we use the same three-signal bucket from Week 1 (declining + real demand +
# page-1/top-3 ranking) as the best available proxy for "actually worth a look."
# This is honest: it's still a proxy, but it's an INDEPENDENT proxy from the
# continuous priority_score, so measuring one against the other is a real check,
# not a tautology.
df["truly_worth_reviewing"] = (
    (df["trend_direction"] == "down") &
    (df["impressions_90d"] >= 300) &
    (df["position_tier"].isin(["page_1", "top_3"]))
).astype(int)

ranked = df.sort_values("priority_score", ascending=False).reset_index(drop=True)

print("precision@K = share of the top-K queue that is 'truly worth reviewing'")
print(f"(baseline: {df['truly_worth_reviewing'].mean()*100:.1f}% of ALL pages are worth reviewing --")
print(" any K below should clear this bar by a wide margin, or the score adds nothing)\n")
for k in [50, 100, 500, 1000]:
    top_k = ranked.head(k)
    precision = top_k["truly_worth_reviewing"].mean()
    print(f"  precision@{k}: {precision*100:.1f}%")

# Because priority_score is built from the SAME three signals as the "truly
# worth reviewing" bucket, precision@K is a trivial 100% at small K -- that's
# expected, not a win, since the two aren't independent yet. The metric only
# becomes a real test once a model has to rank pages the rule bucket is blind
# to (partial matches: declining but just under the demand floor, etc.).
# Show where precision actually starts to fall, to prove the metric can move:
for k in [2000, 4872, 6000, 10000]:
    top_k = ranked.head(k)
    precision = top_k["truly_worth_reviewing"].mean()
    print(f"  precision@{k}: {precision*100:.1f}%")


precision@K = share of the top-K queue that is 'truly worth reviewing'
(baseline: 16.2% of ALL pages are worth reviewing --
 any K below should clear this bar by a wide margin, or the score adds nothing)

  precision@50: 100.0%
  precision@100: 100.0%
  precision@500: 100.0%
  precision@1000: 100.0%
  precision@2000: 100.0%
  precision@4872: 95.8%
  precision@6000: 81.2%
  precision@10000: 48.7%


## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

**One row = one content page** (`content_id`), aggregated over its trailing 90-day window. Confirmed below: 30,000 rows, 30,000 unique `content_id` values, across 32 clients — no page repeats, no pre-aggregation needed.

In [1]:
import pandas as pd

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

# One row = one content page (content_id), aggregated over its trailing 90-day window.
print("Rows:", len(df))
print("Unique content_id:", df["content_id"].nunique(), "-> one row per page, confirmed.")
print("Unique client_id:", df["client_id"].nunique(), "clients in this slice.")
print()

cols = ["content_id", "client_id", "content_type", "main_intent", "word_count",
        "impressions_90d", "clicks_90d", "avg_position", "position_tier",
        "trend_direction", "trend_pct"]
print(df[cols].head(5).to_string(index=False))


Rows: 30000
Unique content_id: 30000 -> one row per page, confirmed.
Unique client_id: 32 clients in this slice.

          content_id         client_id    content_type   main_intent  word_count  impressions_90d  clicks_90d  avg_position position_tier trend_direction  trend_pct
content_304f48230142 client_f369cb89fc keyword article transactional      3221.0             3803          29          10.6      striking            down      -41.4
content_a1fb4e703a9e client_4e07408562 keyword article informational      2481.0            15320           7          20.3      page_3_5            down      -57.7
content_9aa793d4d895 client_7f2253d7e2 keyword article informational      3515.0            12581          11          36.5      page_3_5            down      -60.9
content_331d6c4de07b client_19581e27de keyword article    commercial         NaN            11751          58           6.2        page_1          stable      -13.8
content_d99b7a2d90ca client_3fdba35f04 keyword article inform

## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*

The Week 1 rule and the `priority_score` proxy above both lean on just 3 of the ~40 usable columns in this dataset (trend, demand, position). The code cell below checks whether the *other* signals — freshness, search intent, content type, engagement, CTR, AI-traffic share — carry real information too, among declining pages only.

They do, but **weakly and inconsistently**: freshness tier shifts average priority by ~7 points (declining pages last touched 91–180 days ago actually score *higher* than freshly-updated ones — a mildly counterintuitive pattern a hand-written rule wouldn't guess); intent shifts it by ~2 points (transactional pages slightly outrank informational ones); content type shifts it by ~2 points; and engagement rate, CTR, and AI-traffic share each correlate only weakly (0.08, 0.03, -0.01) with priority on their own.

That's the actual argument for ML over a bigger if-statement: no single one of these signals is strong enough to hand-tune a weight for with confidence, but there are 6+ of them, each nudging the true priority a little, in directions that aren't always obvious (freshness moving the "wrong" way is exactly the kind of interaction a learned model can pick up and a human rule-writer would likely get backwards or ignore). A fixed rule can encode 2–3 strong, obvious signals; it can't responsibly hand-weight six-plus weak, sometimes-surprising ones without just guessing. That gap — more real signal than a person can confidently hand-weight — is where a learned model earns its place, and it's still something to *prove* in Week 4–5 with an actual baseline-vs-model comparison, not assume here.

In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

is_declining = (df["trend_direction"] == "down").astype(int)
demand_component = np.log1p(df["impressions_90d"].clip(lower=0))
demand_component = (demand_component - demand_component.min()) / (demand_component.max() - demand_component.min())
position_value_map = {"top_3": 1.0, "page_1": 0.85, "striking": 0.55, "page_3_5": 0.25, "deep": 0.05}
position_component = df["position_tier"].map(position_value_map).fillna(0.05)
df["priority_score"] = (is_declining * (0.5 * demand_component + 0.5 * position_component) * 100).round(1)

# The Week 1 rule (trend_direction == "down") and this Week 2 3-signal proxy
# only use 3 of the ~40 non-leaky columns available. Do the OTHER signals carry
# real information too? Spot-check a few, among declining pages only:
declining = df[df["trend_direction"] == "down"].copy()

print("Among DECLINING pages, does priority_score vary by other signals a")
print("2-3-line if-statement was never using?\n")

print("-- by freshness_tier (days since last content update) --")
print(declining.groupby("freshness_tier")["priority_score"].mean().round(1).sort_values(ascending=False))

print("\n-- by main_intent --")
print(declining.groupby("main_intent")["priority_score"].mean().round(1).sort_values(ascending=False))

print("\n-- by content_type --")
print(declining.groupby("content_type")["priority_score"].mean().round(1).sort_values(ascending=False))

print("\n-- correlation of priority_score with engagement_rate, ctr, ai_traffic_pct --")
print(declining[["priority_score","engagement_rate","ctr","ai_traffic_pct"]].corr()["priority_score"].round(3))


Among DECLINING pages, does priority_score vary by other signals a
2-3-line if-statement was never using?

-- by freshness_tier (days since last content update) --
freshness_tier
91-180    54.8
0-30      53.4
31-90     49.5
181+      47.7
Name: priority_score, dtype: float64

-- by main_intent --
main_intent
transactional    55.2
navigational     54.8
commercial       54.4
informational    53.5
Name: priority_score, dtype: float64

-- by content_type --
content_type
keyword article       54.0
comparison article    51.8
feedly article        51.6
Name: priority_score, dtype: float64

-- correlation of priority_score with engagement_rate, ctr, ai_traffic_pct --
priority_score     1.000
engagement_rate    0.077
ctr                0.025
ai_traffic_pct    -0.010
Name: priority_score, dtype: float64


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.